# German Unemployment Trends (1991–2025)


### 1. Overview
Since the reunification of Germany, the economic disparity between the former **East (DDR)** and **West** has been a central topic of political and social debate. For decades, the narrative was simple: "The West is rich, the East is catching up."

However, recent data suggests a more complex reality. While the gap has narrowed significantly, unemployment "hotspots" have shifted. It is no longer just an East vs. West story; it is also a story of structural challenges in specific western cities.

### 2. Project Goal
My goal is to tell a data-driven story that answers two key questions:
1.  **The Catch-Up:** Has Eastern Germany successfully closed the gap with the West?
2.  **The New Hotspots:** Which regions are struggling today? (Is the "poor East" narrative still true?)

### 3. Data Source & Methodology

The primary dataset for this analysis is sourced from the **Federal Statistical Office of Germany (Destatis)** and the **Federal Employment Agency (BA)**.

* **Official Source:** [Genesis-Online Database (Table 13211-0007)](https://www-genesis.destatis.de/datenbank/online/statistic/13211/table/13211-0007/chart/map)
* **Dataset Name:** *Registered unemployed, unemployment as a percentage of the civilian labour force, registered vacancies: Länder, years.*
* **Table Code:** `13211-0007`
* **Time Range:** 1991 – 2025 (Annual averages).
* **Granularity:** Federal States (Bundesländer).

**Key Methodological Notes:**
1.  **Hartz IV Reform (2005):** A significant break in the time series occurred in January 2005. The combination of unemployment relief and public assistance led to a statistical increase in registered unemployment numbers. We will mark this event in our visualizations to ensure truthful interpretation.
2.  **Projection Data:** The dataset includes entries for 2025. As the file generation date is listed as January 2026 in the source metadata, these values represent the latest available statistics or finalized annual projections.

**Important Note on Data Quality:**
According to the source documentation, a major methodology change occurred in **January 2005** (combining unemployment relief and public assistance). This explains the visible "spike" in data around 2005, which should be interpreted as a statistical adjustment rather than a sudden economic crash.

---

## Imports and Setup

In [1]:
import pandas as pd
from pathlib import Path

# Resolve paths relative to this notebook's folder.
try:
    BASE_DIR = Path(__vsc_ipynb_file__).parent  # set by VS Code
except NameError:
    BASE_DIR = Path.cwd()  # Jupyter Notebook/Lab start the kernel in the notebook's folder

file_path = BASE_DIR / '13211-0007_en.csv'


## 2. Defining State Names
The raw CSV file does not have a standard "State" column. Instead, the state names appear as headers separating blocks of data. To fix this, we define a list of all 16 German federal states ("Länder") to help our parser identify where each data block begins.

In [2]:
states_list = [
    "Baden-Württemberg", "Bayern", "Berlin", "Brandenburg", "Bremen",
    "Hamburg", "Hessen", "Mecklenburg-Vorpommern", "Niedersachsen",
    "Nordrhein-Westfalen", "Rheinland-Pfalz", "Saarland", "Sachsen",
    "Sachsen-Anhalt", "Schleswig-Holstein", "Thüringen"
]

data = []
current_state = None

**Note:** This list acts as a "filter." When our code reads the file, it will look for these exact names to assign the unemployment data to the correct region.

## 3. Data Extraction and Cleaning
The raw file contains metadata, estimated flags ("e"), and mixed formatting. We cannot use the standard pd.read_csv function here.

**The Logic:**
1. We read the file line by line.
2. If we find a State Name, we store it as the `current_state`.
3. If we find a line starting with a Year (e.g., "1991"), we extract the unemployment rate.
4. We fix the German number format (replacing commas `,` with dots `.`).

In [3]:
def to_float(x):
    x = (x or "").strip()
    if x in {"", "-"}:
        return None
    try:
        return float(x.replace(",", "."))
    except:
        return None

with open(file_path, "r", encoding="utf-8-sig") as f:
    for raw in f:
        line = raw.strip()
        parts = line.split(";")
        first = parts[0].strip()

      
        if first in states_list and len(parts) > 1 and all(p == "" for p in parts[1:]):
            current_state = first
            continue

        # year rows
        if current_state and first.isdigit() and len(first) == 4:
            year = int(first)

      
            v = None
            if len(parts) > 5:
                v = to_float(parts[5])
            if v is None and len(parts) > 3:
                v = to_float(parts[3])

            if v is not None:
                data.append([year, current_state, v])

df = pd.DataFrame(data, columns=["Year", "State", "Unemployment_Rate"])

**Result:** The unstructured text has been successfully converted into a structured DataFrame (df). The loop handled the missing values and format issues automatically.

## 4. Feature Engineering: Adding Regions
To visualize the Convergence story (East vs. West Germany), we need to categorize the states. We create a new column Region.
* **East:** Former DDR states.
* **West:** Former West German states.
* **Berlin:** Treated separately as a unique metropolitan area.

In [4]:

east_states = ["Brandenburg", "Mecklenburg-Vorpommern", "Sachsen", "Sachsen-Anhalt", "Thüringen"]

def get_region(state):
    if state in east_states:
        return "East"
    elif state == "Berlin":
        return "Berlin"
    else:
        return "West"

df["Region"] = df["State"].apply(get_region)

**Analysis:** This step is crucial for our storytelling. Instead of looking at 16 individual lines, we can now aggregate data by these three major groups to see macro-economic trends.

In [5]:
print("Unique states:", df["State"].nunique())
missing = sorted(set(states_list) - set(df["State"].unique()))
print("Missing states:", missing)

df.to_csv(BASE_DIR / "german_unemployment_data.csv", index=False)
df.head()

Unique states: 16
Missing states: []


,Year,State,Unemployment_Rate,Region
0,1991,Baden-Württemberg,3.7,West
1,1992,Baden-Württemberg,4.4,West
2,1993,Baden-Württemberg,6.3,West
3,1994,Baden-Württemberg,7.5,West
4,1995,Baden-Württemberg,7.4,West


### **Data Quality Summary:**

* **Readiness:** The data is clean and ready for visualization.